# acoustic-watermark-lab — try it yourself

Hide an **encrypted message inside your own song**, then prove it **survives a
lossy re-encode** (what WhatsApp does). Everything runs here in Google Colab —
nothing is installed on your machine and your audio never leaves this session.
Run the cells top to bottom (Shift+Enter).


### 1. Setup

In [ ]:
!pip -q install numpy cryptography imageio-ffmpeg

### 2. Load the engine
These cells write the project's modules into this session, then import them.

In [ ]:
%%writefile rs.py
"""Reed-Solomon over GF(256), systematic. Reference implementation.

Kept dependency-free and deliberately simple so the JavaScript port in the
PWA can mirror it line for line.
"""

GF_EXP = [0] * 512
GF_LOG = [0] * 256

def _init_tables(primitive=0x11d):
    x = 1
    for i in range(255):
        GF_EXP[i] = x
        GF_LOG[x] = i
        x <<= 1
        if x & 0x100:
            x ^= primitive
    for i in range(255, 512):
        GF_EXP[i] = GF_EXP[i - 255]

_init_tables()


def gf_mul(a, b):
    if a == 0 or b == 0:
        return 0
    return GF_EXP[GF_LOG[a] + GF_LOG[b]]


def gf_div(a, b):
    if b == 0:
        raise ZeroDivisionError
    if a == 0:
        return 0
    return GF_EXP[(GF_LOG[a] - GF_LOG[b]) % 255]


def gf_pow(a, n):
    return GF_EXP[(GF_LOG[a] * n) % 255]


def gf_inv(a):
    return GF_EXP[(255 - GF_LOG[a]) % 255]


def poly_mul(p, q):
    r = [0] * (len(p) + len(q) - 1)
    for i, pi in enumerate(p):
        if pi == 0:
            continue
        for j, qj in enumerate(q):
            if qj:
                r[i + j] ^= gf_mul(pi, qj)
    return r


def poly_eval(p, x):
    y = 0
    for c in p:
        y = gf_mul(y, x) ^ c
    return y


def generator_poly(nsym):
    g = [1]
    for i in range(nsym):
        g = poly_mul(g, [1, gf_pow(2, i)])
    return g


def rs_encode(data, nsym):
    """Append nsym parity bytes to data."""
    gen = generator_poly(nsym)
    out = list(data) + [0] * nsym
    for i in range(len(data)):
        coef = out[i]
        if coef == 0:
            continue
        for j in range(1, len(gen)):
            out[i + j] ^= gf_mul(gen[j], coef)
    return bytes(data) + bytes(out[len(data):])


def _syndromes(msg, nsym):
    return [poly_eval(msg, gf_pow(2, i)) for i in range(nsym)]


def _berlekamp_massey(synd, nsym):
    err_loc = [1]
    old_loc = [1]
    for i in range(nsym):
        old_loc = old_loc + [0]
        delta = synd[i]
        for j in range(1, len(err_loc)):
            delta ^= gf_mul(err_loc[len(err_loc) - 1 - j], synd[i - j])
        if delta != 0:
            if len(old_loc) > len(err_loc):
                new_loc = [gf_mul(c, delta) for c in old_loc]
                old_loc = [gf_mul(c, gf_inv(delta)) for c in err_loc]
                err_loc = new_loc
            scale = [gf_mul(c, delta) for c in old_loc]
            err_loc = [
                (err_loc[len(err_loc) - 1 - k] if k < len(err_loc) else 0)
                ^ (scale[len(scale) - 1 - k] if k < len(scale) else 0)
                for k in range(max(len(err_loc), len(scale)))
            ][::-1]
    while err_loc and err_loc[0] == 0:
        err_loc.pop(0)
    return err_loc


def _find_errors(err_loc, nmess):
    errs = len(err_loc) - 1
    pos = []
    for i in range(nmess):
        if poly_eval(err_loc, gf_pow(2, 255 - i)) == 0:
            pos.append(nmess - 1 - i)
    if len(pos) != errs:
        raise ValueError("RS: could not locate errors")
    return pos


def _correct(msg, synd, pos):
    coef_pos = [len(msg) - 1 - p for p in pos]

    # Error locator from the found positions.
    e_loc = [1]
    for i in coef_pos:
        e_loc = poly_mul(e_loc, [gf_pow(2, i), 1])

    # Error evaluator.
    rsynd = synd[::-1]
    ee = poly_mul(rsynd, e_loc)
    ee = ee[len(ee) - len(coef_pos):]

    # Formal derivative of the locator.
    e_loc_prime = e_loc[len(e_loc) % 2::2]

    out = list(msg)
    for i, p in enumerate(coef_pos):
        xi = gf_pow(2, p)
        xi_inv = gf_inv(xi)
        num = poly_eval(ee, xi_inv)
        den = poly_eval(e_loc_prime, gf_mul(xi_inv, xi_inv))
        if den == 0:
            raise ValueError("RS: undefined error magnitude")
        mag = gf_mul(xi, gf_div(num, den))
        out[pos[i]] ^= mag
    return bytes(out)


def rs_decode(msg, nsym):
    """Correct up to nsym//2 byte errors. Returns the data portion."""
    msg = list(msg)
    synd = _syndromes(msg, nsym)
    if max(synd) == 0:
        return bytes(msg[:-nsym])
    err_loc = _berlekamp_massey(synd, nsym)
    if len(err_loc) - 1 > nsym // 2:
        raise ValueError("RS: too many errors to correct")
    pos = _find_errors(err_loc, len(msg))
    fixed = _correct(msg, synd, pos)
    if max(_syndromes(list(fixed), nsym)) != 0:
        raise ValueError("RS: correction failed")
    return bytes(fixed[:-nsym])


In [ ]:
%%writefile codec.py
"""Framing helpers: how the encrypted body is split into Reed-Solomon blocks
and interleaved so a burst of codec damage is spread across many blocks.

Pure integer/byte arithmetic, no dependencies.
"""

# Reed-Solomon block size (data bytes per block, before parity).
CHUNK = 150


def parity_for(k):
    """Parity bytes for a block of k data bytes, about a quarter rate.

    Integer arithmetic only, and no rounding function: rounding differs between
    languages (Python rounds 14.5 down, JavaScript up), and both ends must agree
    to the byte or the receiver reads the wrong number of symbols.
    """
    return max(10, min(64, 2 * -(-k // 8)))


def chunk_sizes(total):
    """How a body of `total` bytes is split into Reed-Solomon blocks.

    Both ends derive this from the length in the header, so no block boundaries
    need to be transmitted.
    """
    out = []
    left = total
    while left > 0:
        n = min(CHUNK, left)
        out.append(n)
        left -= n
    return out


def coded_size(total):
    return sum(n + parity_for(n) for n in chunk_sizes(total))


def interleave(data, depth=8):
    """Spread a burst of damage across separate Reed-Solomon positions."""
    if len(data) <= depth:
        return data
    out = bytearray()
    for r in range(depth):
        out += data[r::depth]
    return bytes(out)


def deinterleave(data, depth=8):
    if len(data) <= depth:
        return data
    n = len(data)
    out = [0] * n
    pos = 0
    for r in range(depth):
        idxs = range(r, n, depth)
        for j, idx in enumerate(idxs):
            out[idx] = data[pos + j]
        pos += len(idxs)
    return bytes(out)


In [ ]:
%%writefile room.py
"""Room crypto: turn a room name + password into a key, and seal / unseal the
payload with AES-GCM. Plus the Reed-Solomon body coding used by the watermark.

The room name is the salt, so the same password in two different rooms yields
two unrelated keys and nothing is precomputable across rooms.
"""

import hashlib
import os

from cryptography.hazmat.primitives.ciphers.aead import AESGCM

import codec
from rs import rs_decode, rs_encode

MAGIC = 0xC7
VERSION = 1
KDF_ROUNDS = 600_000


def room_key(room, password):
    salt = hashlib.sha256(
        b"awl-v1:room:" + room.strip().lower().encode("utf-8")
    ).digest()
    return hashlib.pbkdf2_hmac(
        "sha256", password.encode("utf-8"), salt, KDF_ROUNDS, 32
    )


def encode_body(body):
    """Reed-Solomon code the body, block by block."""
    out = bytearray()
    pos = 0
    for n in codec.chunk_sizes(len(body)):
        out += rs_encode(body[pos:pos + n], codec.parity_for(n))
        pos += n
    return bytes(out)


def decode_body(coded, total):
    out = bytearray()
    pos = 0
    for n in codec.chunk_sizes(total):
        width = n + codec.parity_for(n)
        out += rs_decode(coded[pos:pos + width], codec.parity_for(n))
        pos += width
    return bytes(out)


def sealb(data, room, password):
    """Encrypt raw bytes (a compressed payload) for the given room + password."""
    key = room_key(room, password)
    nonce = os.urandom(12)
    ct = AESGCM(key).encrypt(nonce, bytes(data), bytes([MAGIC, VERSION]))
    return nonce + ct


def unsealb(blob, room, password):
    if len(blob) < 28:
        raise ValueError("too small")
    key = room_key(room, password)
    try:
        return AESGCM(key).decrypt(blob[:12], blob[12:], bytes([MAGIC, VERSION]))
    except Exception:
        raise ValueError("Wrong room or wrong password")


In [ ]:
%%writefile cover.py
# -*- coding: utf-8 -*-
"""Hide an encrypted message inside a file the user brings - a song, a voice
clip - at full fidelity, and read it back.

Nothing is appended and no marker is added to the container: the file stays a
normal audio file to anything that inspects it. The message lives *in the sound*,
as a tiny change to the energy balance between two mid-range bands (1.2-2.4 vs
2.4-3.6 kHz) - the region a lossy codec must keep, so it survives being shared.

The rest of the spectrum (bass, highs) and the stereo image are left untouched,
and the file keeps its full length: only the span the message needs is marked,
everything after it is copied through as-is.

Built from three small pieces: the crypto (room.py), Reed-Solomon (rs.py) and
the framing helpers (codec.py).
"""

import hashlib
import os
import struct
import subprocess
import sys
import tempfile
import wave
import zlib

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import codec
import room as roomlib
from rs import rs_encode, rs_decode

RATE = 44100
FN = 2048
FH = 2048
REPEAT = 4                       # per-copy repeat; the whole-file tiling adds more
DELTA = 0.22                     # small step: the whole-file redundancy pays for it
MIN_VOTES = 8                    # required repetitions of each bit across the file
BAND = (1200, 2400, 2400, 3600)
MARK = b"\x9a\x53"
HDR_PARITY = 10
HDR_LEN = 4 + HDR_PARITY


def ffmpeg():
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


# ---- framing (identical to wm_full, so the two can interoperate) ----
def _bins():
    b1lo, b1hi, b2lo, b2hi = BAND
    return ((round(b1lo * FN / RATE), round(b1hi * FN / RATE)),
            (round(b2lo * FN / RATE), round(b2hi * FN / RATE)))


def _dither(room, pw, n):
    """A secret per-frame offset of the QIM grid, in [0, DELTA).

    This is what makes the mark undetectable. Plain QIM snaps d = 0.5 ln(E1/E2)
    to fixed multiples of DELTA, so frac(d/DELTA) collapses to 0 on every
    watermarked frame - a spike any steganalyst sees at once. Here the grid is
    shifted by delta_i on frame i, and delta_i is derived from the room+password.
    To anyone without the key, frac(d/DELTA) = frac(delta_i/DELTA) is spread
    uniformly across [0,1): no spike, and no way even to tell a mark is present.
    We subtract delta_i back before reading, so decoding is unaffected.
    """
    seed = int.from_bytes(
        hashlib.sha256(("awl-dither|" + room + "|" + pw).encode()).digest()[:8],
        "big")
    return np.random.default_rng(seed).random(n) * DELTA


def _pack(text):
    raw = text.encode("utf-8")
    comp = zlib.compress(raw, 9)
    return bytes([1]) + comp if len(comp) < len(raw) else bytes([0]) + raw


def _unpack(data):
    return zlib.decompress(data[1:]).decode("utf-8") if data[0] == 1 else data[1:].decode("utf-8")


def _payload_bits(text, room, pw):
    body = roomlib.sealb(_pack(text), room, pw)
    coded = codec.interleave(roomlib.encode_body(body))
    payload = rs_encode(MARK + struct.pack(">H", len(body)), HDR_PARITY) + coded
    return np.unpackbits(np.frombuffer(payload, np.uint8))


def samples_needed(text, room, pw):
    return len(_payload_bits(text, room, pw)) * MIN_VOTES * FH + FN


# ---- QIM over the WHOLE file, key-dithered grid ----
def _embed(sig, bits, room, pw):
    """Watermark every frame of the signal.

    The payload is tiled across the whole file: frame j carries
    bit[(j // REPEAT) % len(bits)]. Two things follow. There is no unmarked
    stretch an attacker could hold up as an internal reference, and every bit is
    repeated many times over the file, so the per-frame nudge (DELTA) can stay
    small - which is what keeps the mark below a steganalyst's second-order tests.
    """
    g1, g2 = _bins()
    F = (len(sig) - FN) // FH + 1
    dith = _dither(room, pw, F)
    nb = len(bits)
    out = sig.copy()
    for j in range(F):
        bit = int(bits[(j // REPEAT) % nb])
        pos = j * FH
        S = np.fft.rfft(out[pos:pos + FN])
        E1 = np.sum(np.abs(S[g1[0]:g1[1]])**2) + 1e-12
        E2 = np.sum(np.abs(S[g2[0]:g2[1]])**2) + 1e-12
        d = 0.5 * np.log(E1 / E2)
        delta = dith[j]                            # secret shift of the grid
        q = int(round((d - delta) / DELTA))
        if q % 2 != bit:
            q += 1 if (d - delta) >= q * DELTA else -1
        adj = (q * DELTA + delta) - d              # snap to the shifted grid
        S[g1[0]:g1[1]] *= np.exp(adj / 2)
        S[g2[0]:g2[1]] *= np.exp(-adj / 2)
        out[pos:pos + FN] = np.fft.irfft(S, FN)
    return out


def _frame_bits(x, off, dith, nframes):
    """Raw per-frame bit read for `nframes` frames starting at `off`."""
    g1, g2 = _bins()
    out = np.empty(nframes, np.int8)
    for j in range(nframes):
        pos = off + j * FH
        if pos + FN > len(x):
            return out[:j]
        S = np.fft.rfft(x[pos:pos + FN])
        E1 = np.sum(np.abs(S[g1[0]:g1[1]])**2) + 1e-12
        E2 = np.sum(np.abs(S[g2[0]:g2[1]])**2) + 1e-12
        out[j] = int(round((0.5 * np.log(E1 / E2) - dith[j]) / DELTA)) % 2
    return out


# ---- audio I/O via ffmpeg (any input -> 44.1k stereo float) ----
def _load_stereo(path):
    raw = subprocess.run(
        [ffmpeg(), "-v", "error", "-i", path, "-ac", "2", "-ar", str(RATE),
         "-f", "s16le", "-"], capture_output=True).stdout
    if not raw:
        raise ValueError("ffmpeg could not read this file as audio")
    return np.frombuffer(raw, np.int16).astype(np.float64).reshape(-1, 2) / 32768.0


def _load_mono(path):
    raw = subprocess.run(
        [ffmpeg(), "-v", "error", "-i", path, "-ac", "1", "-ar", str(RATE),
         "-f", "s16le", "-"], capture_output=True).stdout
    if not raw:
        raise ValueError("ffmpeg could not read this file as audio")
    return np.frombuffer(raw, np.int16).astype(np.float64) / 32768.0


def _write_stereo_to(out_path, st, fmt):
    """Encode full-length stereo to the chosen output format."""
    with tempfile.TemporaryDirectory() as td:
        wav = os.path.join(td, "full.wav")
        with wave.open(wav, "wb") as w:
            w.setnchannels(2)
            w.setsampwidth(2)
            w.setframerate(RATE)
            w.writeframes((np.clip(st, -1, 1) * 32767).astype(np.int16).reshape(-1).tobytes())
        if fmt == "wav":
            os.replace(wav, out_path)
            return
        args = {"mp3": ["-c:a", "libmp3lame", "-b:a", "320k"],
                "flac": ["-c:a", "flac"]}.get(fmt, ["-c:a", "libmp3lame", "-b:a", "320k"])
        subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-i", wav] + args + [out_path],
                       check=True)


# ---- public API ----
def encode_cover(in_path, text, room, pw, out_path, out_fmt="mp3"):
    """Watermark `in_path` with `text`; write the full-length result to out_path.

    The message is embedded into the mid channel (L+R) across the whole file, so
    only the 1.2-3.6 kHz band moves and the file keeps its length and stereo
    image. Returns (seconds_marked, total_seconds); marked == total here.
    """
    bits = _payload_bits(text, room, pw)
    nb = len(bits)

    st = _load_stereo(in_path)
    F = (len(st) - FN) // FH + 1
    if F < nb * MIN_VOTES:
        need_s = nb * MIN_VOTES * FH / RATE
        raise ValueError(
            f"message too long for this file: it needs at least {need_s:.0f}s of "
            f"audio but the file is {len(st)/RATE:.0f}s. Use a longer file or "
            f"shorter text.")

    L = st[:, 0]
    R = st[:, 1]
    mid = (L + R) / 2.0
    side = (L - R) / 2.0
    mid2 = _embed(mid, bits, room, pw)

    out = st.copy()
    out[:, 0] = mid2 + side
    out[:, 1] = mid2 - side
    _write_stereo_to(out_path, out, out_fmt)
    total = len(st) / RATE
    return total, total


def decode_cover(path, room, pw, max_off=9000):
    """Read the hidden text from a watermarked file, or raise ValueError.

    The grid is dithered by the key, so the header itself only lines up for the
    right room+password: a wrong key matches nothing, reads no bodies, and fails
    fast - and it cannot even tell whether a watermark was present. That is why
    the error is deliberately the same whether the file is clean or the key is
    wrong; distinguishing them would leak that a message exists.
    """
    x = _load_mono(path)                                # mono downmix == mid channel
    hdr_frames = HDR_LEN * 8 * REPEAT
    for off in range(0, max_off, 64):
        F = (len(x) - off - FN) // FH + 1
        if F < hdr_frames:
            break
        dith = _dither(room, pw, F)
        # find the frame alignment from the header (first copy), cheaply
        hfb = _frame_bits(x, off, dith, hdr_frames)
        if len(hfb) < hdr_frames:
            break
        hbits = np.array(
            [1 if hfb[i*REPEAT:(i+1)*REPEAT].sum() * 2 >= REPEAT else 0
             for i in range(HDR_LEN * 8)], np.uint8)
        try:
            header = rs_decode(np.packbits(hbits).tobytes(), HDR_PARITY)
        except Exception:
            continue
        if header[:2] != MARK:
            continue
        blen = struct.unpack(">H", header[2:4])[0]
        if not (28 <= blen <= 4000):
            continue
        # a real frame: now vote each bit across every copy in the whole file
        nb = (HDR_LEN + codec.coded_size(blen)) * 8
        fb = _frame_bits(x, off, dith, F)
        idx = (np.arange(len(fb)) // REPEAT) % nb
        votes = np.zeros(nb); cnt = np.zeros(nb)
        np.add.at(votes, idx, fb)
        np.add.at(cnt, idx, 1)
        bits = (votes * 2 >= cnt).astype(np.uint8)
        try:
            body = roomlib.decode_body(
                codec.deinterleave(np.packbits(bits).tobytes()[HDR_LEN:]), blen)
            return _unpack(roomlib.unsealb(body, room, pw))
        except Exception:
            continue
    raise ValueError("no hidden message here, or wrong room/password")


def capacity_chars(seconds):
    """Rough guide: how many bytes of message an audio of this length can hold."""
    frames = int((seconds * RATE - FN) / FH)
    usable_bits = frames // MIN_VOTES                  # each bit repeated MIN_VOTES times
    body_bytes = usable_bits // 8 - HDR_LEN
    # framing (RS + interleave) roughly doubles the sealed body; sealing adds ~28
    return max(0, body_bytes // 2 - 28)


if __name__ == "__main__":
    # offline self-test: pass a path to any audio file (a song, a voice clip)
    #   python cover.py path/to/audio.mp3
    if len(sys.argv) < 2:
        print("usage: python cover.py <audio-file>")
        sys.exit(1)
    song = sys.argv[1]
    msg = sys.argv[2] if len(sys.argv) > 2 else "meet at 8 by the kiosk"
    room, pw = "room1", "password1"
    with tempfile.TemporaryDirectory() as td:
        out = os.path.join(td, "wm.mp3")
        marked, total = encode_cover(song, msg, room, pw, out)
        print(f"encoded: marked {marked:.0f}s of a {total:.0f}s file -> {os.path.getsize(out)//1024} KB mp3")
        got = decode_cover(out, room, pw)
        print(f"decoded: {got!r}  {'OK' if got == msg else 'MISMATCH'}")
        try:
            decode_cover(out, room, "wrong")
            print("wrong password: LEAK (bad)")
        except ValueError:
            print("wrong password: rejected OK")


In [ ]:
import cover
print('engine loaded — rate', cover.RATE, 'Hz, band', cover.BAND, 'Hz')

### 3. Upload a song
Pick an audio file of **a few minutes** (a short clip cannot hold much text).
Nothing copyrighted is stored — it stays in this temporary session.

In [ ]:
from google.colab import files
up = files.upload()
SONG = next(iter(up))
print('uploaded:', SONG)

### 4. Your secret
The room + password are the key. Only they reveal the message.

In [ ]:
ROOM = "room1"
PASSWORD = "correct horse battery staple"
MESSAGE = "meet at 8 by the kiosk"

### 5. Hide it
Only the 1.2–3.6 kHz energy ratio moves; the file keeps its full length and sounds identical. Output is a normal MP3.

In [ ]:
marked, total = cover.encode_cover(SONG, MESSAGE, ROOM, PASSWORD, 'hidden.mp3')
print(f'watermarked the first {marked:.0f}s of {total:.0f}s -> hidden.mp3')
# from google.colab import files as _f; _f.download('hidden.mp3')  # grab it if you like

### 6. Reveal it
Right key returns the text; a wrong key returns nothing.

In [ ]:
print('correct key :', cover.decode_cover('hidden.mp3', ROOM, PASSWORD))
try:
    cover.decode_cover('hidden.mp3', ROOM, 'wrong password')
    print('wrong key   : LEAK (should not happen)')
except ValueError:
    print('wrong key   : rejected')

### 7. Prove it survives WhatsApp-grade compression
Re-encode to Opus and AAC at 128 kbps (what a messenger does) and decode again.

In [ ]:
import subprocess
FF = cover.ffmpeg()
for name, args in [('Opus 128k', ['-c:a','libopus','-b:a','128k']),
                   ('AAC 128k',  ['-c:a','aac','-b:a','128k'])]:
    out = 'ch.' + ('ogg' if 'libopus' in args else 'm4a')
    subprocess.run([FF,'-y','-loglevel','error','-i','hidden.mp3']+args+[out], check=True)
    got = cover.decode_cover(out, ROOM, PASSWORD)
    print(f'{name:10} -> ' + ('SURVIVED: '+got if got==MESSAGE else 'lost'))

### What you just saw

Your message hides in your song, the file sounds identical, and it **survives
Opus/AAC re-encoding** — so it comes back intact after being shared.

The payload is spread across the whole file at a small embedding strength, so
there is no clean stretch to compare against and the per-frame change is tiny.
Measured against a real song, the watermarked file matches the clean original on
the grid test and on an intra-file autocorrelation test.

**Honest limitation:** a small residual difference in overall temporal correlation
remains (about 0.44 vs 0.51), so a more sensitive or machine-learning detector
might still separate them. The honest claim is "defeats the first- and
second-order tests here", not "undetectable". A full steganalysis evaluation is
the next step.
